# TalentDesk, Module 2 Section 3 Lab (Exercise): Grep, Glob, Edit, and the Read+Write Fallback

A hands-on exercise on Claude Code's **built-in code tools**. It combines the two M2S3 skills: using
**Grep** to find content, **Glob** to find files, and **Edit** for a surgical change that needs a
**unique** anchor (Lab 1), then falling back to **Read plus Write** when an anchor cannot be unique and
exploring a codebase **incrementally** to keep context lean (Lab 2). You fill in four short `TODO`
blocks over a small TalentDesk sample repo; everything else is provided. The tool analogues are all
testable offline, and a live **Claude Agent SDK** run uses the real built-in tools. Runs **Sonnet**
(`claude-sonnet-4-6`).

## The real-world scenario

TalentDesk's rules live in code: a `talentdesk` package with candidate, screening, and scheduling
modules. To change one line of the screening bar, the agent should not read the whole codebase. It
**searches** for where the rule lives (Grep), **discovers** the right files by name (Glob), then makes
a **surgical** change (Edit). Each tool does one job.

Two things complicate that. Sometimes the text you want to change appears several times, so a
single-anchor Edit refuses rather than risk editing the wrong line; then you fall back to **Read plus
Write**. And sometimes you just need to understand how the screening decision flows before touching it,
which you do **incrementally**, reading only the slices you need.

The question this lab answers: **which built-in tool does what, why does a targeted Edit require a
unique anchor, and what do you do when it cannot be unique or when you need to explore a big
codebase?**

## Objectives

- Use **Grep** (content search) to locate functions, and **Glob** (path patterns) to find tests and
  config.
- Use **Edit** for a targeted change, and see why an **ambiguous anchor** must fail.
- Fall back to **Read then modify then Write** when an anchor cannot be made unique.
- Explore **incrementally** (Grep to Read to trace) and keep the context window small.

## The outcome you should reach

By the end you will have:

- a Grep that returns the file and line of every match, and a Glob that finds tests and config;
- an Edit that accepts a unique anchor and rejects an ambiguous one;
- a Read+Write fallback that applies a change Edit refused;
- and an incremental trace of the screening path that reads a small fraction of the repo.

Target time: **20 to 30 minutes.** Four small `TODO` blocks, all testable offline. The live built-in
tools need a real key and Node.js 18+.

## How to run

Run top to bottom. The repo creation and the tool-analogue cells run anywhere. The live cell calls
Claude with the real built-in tools, so paste a real key into **Setup 2/3** and re-run from the top;
**Node.js 18+** must be installed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages. The live cell uses the **Agent SDK** to drive Claude Code's
built-in tools; the offline cells use only Python. The Agent SDK also needs Node.js 18+, which cannot
be pip-installed.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` for the live
cell.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # filesystem paths for the sample repo
import re                                       # our offline Grep uses regex
import sys                                       # detect Windows (it needs a special event loop)
import textwrap                                  # keeps the embedded file bodies readable
from pathlib import Path                          # our offline Glob uses pathlib
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv             #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cell will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}
    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:    box["value"] = loop.run_until_complete(make_coro())
        except Exception as e: box["error"] = e
        finally: loop.close()
    t = threading.Thread(target=worker); t.start(); t.join()
    if "error" in box: raise box["error"]
    return box.get("value")

print("live model calls:", "ON" if RUN_LIVE else "OFF (tool analogues run offline)")

**This cell:** writes a small **TalentDesk sample repo** to disk: a package with candidate,
screening, and scheduling modules, a couple of tests, and config files. Everything below searches and
edits these real files. The screening path (`screen_candidate` calls `meets_bar` calls `_lookup`) is
what you will trace and edit.

In [ ]:
# ===== SETUP 3/3 - create the sample repo =====
REPO = os.path.join(os.getcwd(), "talentdesk_repo")   # the sample codebase root

FILES = {
    "talentdesk/__init__.py": "",
    "talentdesk/candidates.py": textwrap.dedent("""\
        CANDIDATES = {
            "C1": {"stage": 2, "years": 5},
            "C2": {"stage": 3, "years": 1},
        }
        STAGE_NAMES = {1: "applied", 2: "screening", 3: "interview", 4: "offer"}

        def get_candidate_stage(candidate_id):
            candidate = _lookup(candidate_id)
            if candidate is None:
                return None
            return STAGE_NAMES[candidate["stage"]]

        def _lookup(candidate_id):
            return CANDIDATES.get(candidate_id)
        """),
    "talentdesk/screening.py": textwrap.dedent("""\
        from talentdesk.candidates import _lookup

        MIN_YEARS = 3

        def meets_bar(candidate_id):
            candidate = _lookup(candidate_id)
            if candidate is None:
                return None
            return candidate["years"] >= MIN_YEARS

        def screen_candidate(candidate_id):
            eligible = meets_bar(candidate_id)
            if eligible is None:
                return None
            return "advance " + candidate_id if eligible else "reject: below bar"
        """),
    "talentdesk/scheduling.py": 'def get_slot(candidate_id):\n    return "slot for " + candidate_id\n',
    "tests/test_candidates.py": ('from talentdesk.candidates import get_candidate_stage\n'
                                 'def test_stage():\n    assert get_candidate_stage("C1") == "screening"\n'),
    "tests/test_screening.py": ('from talentdesk.screening import screen_candidate\n'
                               'def test_screen():\n    assert screen_candidate("C1").startswith("advance")\n'),
    "config/settings.toml": '[talentdesk]\nmin_years = 3\n',
    "pyproject.toml": '[tool.pytest.ini_options]\ntestpaths = ["tests"]\n',
    "README.md": "# TalentDesk sample repo\n",
}
for rel, content in FILES.items():
    path = os.path.join(REPO, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)
print("created repo at", REPO, "with", len(FILES), "files")

### Four tools, four jobs

**Grep** searches file *contents* by pattern ("which file defines `screen_candidate`?"). **Glob**
searches file *paths* by pattern ("where are the tests?", `**/test_*.py`). **Read** loads one file or a
slice. **Edit** makes a targeted change by replacing a uniquely matching anchor; **Write** overwrites a
whole file.

The rule that trips people up: **Edit's anchor must be unique in the file.** If the text appears more
than once, Edit fails rather than risk changing the wrong one. When an anchor cannot be made unique,
that is the signal to fall back to Read plus Write.

---

### 🎯 Part A - search, discover, edit

**This cell:** an offline **Glob** (provided): match files by path pattern, without opening
them.

In [ ]:
# ===== Glob: find files by path pattern (provided) =====
def glob_files(pattern):                           # pattern -> sorted matching paths (relative)
    return sorted(str(p.relative_to(REPO)) for p in Path(REPO).glob(pattern) if p.is_file())

print("tests  :", glob_files("**/test_*.py"))
print("configs:", glob_files("**/*.toml"))

**TODO 1 (about 5 minutes).** Complete `grep()`, the content search. Scan each file matching
`file_glob`, and for every line that matches the regex, collect a `(relative_path, line_number,
stripped_text)` tuple. Line numbers start at 1.

In [ ]:
# ===== TODO 1 - Grep: find content across files =====
def grep(pattern, file_glob="**/*.py"):            # pattern -> list of (path, line_no, text)
    rx = re.compile(pattern)
    hits = []
    for p in Path(REPO).glob(file_glob):
        if not p.is_file():
            continue
        for i, line in enumerate(p.read_text().splitlines(), 1):   # 1-based line numbers
            # 👉 TODO 1: if rx.search(line), append (str(p.relative_to(REPO)), i, line.strip()) to hits
            pass
    return hits

for hit in grep(r"def screen_candidate"):
    print("  ", hit)
for hit in grep(r"MIN_YEARS"):
    print("  ", hit)

**Self-check (offline).**

In [ ]:
# ===== self-check for TODO 1 =====
sc = grep(r"def screen_candidate")
assert len(sc) == 1 and sc[0][0] == "talentdesk/screening.py", "should find the one definition"
assert len(grep(r"MIN_YEARS")) == 2, "MIN_YEARS appears twice (assignment + use)"
assert grep(r"def _lookup")[0][0] == "talentdesk/candidates.py", "the def lives in candidates.py"
print("TODO 1 checks passed")

**This cell:** offline **Read** helpers (provided): a numbered view for inspection and a plain
slice for editing. You Read before editing (the read-before-edit rule).

In [ ]:
# ===== Read: load a file (or a slice) (provided) =====
def read_numbered(rel, start=1, end=None):         # rel path -> numbered lines from start..end
    lines = open(os.path.join(REPO, rel)).read().splitlines()
    end = end or len(lines)
    return "\n".join(f"{i:4}  {lines[i-1]}" for i in range(start, min(end, len(lines)) + 1))

def read_file(rel, start=1, end=None):             # rel path -> plain text slice (no line numbers)
    lines = open(os.path.join(REPO, rel)).read().splitlines()
    end = end or len(lines)
    return "\n".join(lines[start - 1:end])

print(read_numbered("talentdesk/screening.py", 1, 3))

**TODO 2 (about 5 minutes).** Complete `edit()`, which enforces the uniqueness rule. Count how
many times the anchor appears: 0 means not found; more than 1 means ambiguous (refuse); exactly 1 means
apply the replacement. Return a clear message in each case, and never guess.

In [ ]:
# ===== TODO 2 - Edit: replace a UNIQUE anchor (fails on ambiguity) =====
def edit(rel, old, new):                           # targeted replacement with the uniqueness rule
    path = os.path.join(REPO, rel)
    text = open(path).read()
    n = text.count(old)                             # how many times the anchor appears
    # 👉 TODO 2a: if n == 0, return f"FAILED: anchor not found in {rel}"
    # 👉 TODO 2b: if n > 1, return f"FAILED: anchor appears {n} times in {rel}; add context"
    # 👉 TODO 2c: otherwise write text.replace(old, new) back and return f"OK: edited {rel}"
    return "TODO: implement edit"

**Self-check (offline).** A unique anchor (`MIN_YEARS = 3`) should edit; an ambiguous one
(`return None`, which appears twice in `screening.py`) should be refused.

In [ ]:
# ===== self-check for TODO 2 =====
assert edit("talentdesk/screening.py", "MIN_YEARS = 3", "MIN_YEARS = 5").startswith("OK"), "unique anchor edits"
assert "MIN_YEARS = 5" in open(os.path.join(REPO, "talentdesk/screening.py")).read(), "the change landed"
res = edit("talentdesk/screening.py", "return None", 'return "unknown candidate"')
assert res.startswith("FAILED") and "2 times" in res, "ambiguous anchor must be refused"
assert edit("talentdesk/screening.py", "not_present_anywhere", "x").startswith("FAILED")
print("TODO 2 checks passed")

---

### 🎯 Part B - fall back gracefully, explore incrementally

When an anchor cannot be made unique (and you want to change every occurrence), fall back to **Read
then modify then Write**: load the whole file, transform it in memory, and write it back.

In [ ]:
# ===== Write: overwrite a whole file (the fallback's tool, provided) =====
def write_file(rel, content):
    open(os.path.join(REPO, rel), "w").write(content)
    return f"OK: wrote {rel}"
print("write_file ready")

**TODO 3 (about 4 minutes).** Complete `apply_fallback()`, the Read+Write fallback. Read the whole
file, replace **every** occurrence of `old` with `new` in memory, then write it back. This handles the
multi-occurrence change that `edit()` refused.

In [ ]:
# ===== TODO 3 - Read + modify + Write fallback =====
def apply_fallback(rel, old, new):                 # change every occurrence Edit could not do safely
    # 👉 TODO 3a: text = read_file(rel)                       # READ the whole file
    # 👉 TODO 3b: text = text.replace(old, new)               # MODIFY all occurrences in memory
    # 👉 TODO 3c: return write_file(rel, text + "\n")         # WRITE it back
    return "TODO: implement apply_fallback"

**Self-check (offline).** The ambiguous `return None` change that Edit refused should now apply to
both occurrences.

In [ ]:
# ===== self-check for TODO 3 =====
path = os.path.join(REPO, "talentdesk/screening.py")
assert open(path).read().count("return None") == 2, "two occurrences before the fallback"
apply_fallback("talentdesk/screening.py", "return None", 'return "unknown candidate"')
after = open(path).read()
assert after.count("return None") == 0, "the fallback replaced every occurrence"
assert after.count('return "unknown candidate"') == 2, "both became the new text"
print("TODO 3 checks passed")

**This cell:** the `first_hit()` helper (provided): the file and line of the first Grep match, used
to trace one function to the next.

In [ ]:
# ===== first_hit: the first grep match (provided) =====
def first_hit(pattern):
    hits = grep(pattern)
    return hits[0] if hits else None
print("first_hit ready")

**TODO 4 (about 5 minutes).** Complete `incremental_trace()`. Follow the screening path defined in
`CHAIN`: for each function, find it with `first_hit`, read just its slice with `read_file`, add the
slice's length to `read_bytes`, and record which file it lives in. Reading only these slices is what
keeps context small.

In [ ]:
# ===== TODO 4 - trace screen_candidate -> meets_bar -> _lookup, one slice at a time =====
CHAIN = [(r"def screen_candidate", 4), (r"def meets_bar", 4), (r"def _lookup", 1)]   # (pattern, slice span)

def incremental_trace():                           # -> (trace list, total bytes read)
    read_bytes = 0
    trace = []
    for pattern, span in CHAIN:
        hit = first_hit(pattern)                    # (file, line, text)
        f, line, _ = hit
        # 👉 TODO 4a: chunk = read_file(f, line, line + span)   # read just this function's slice
        # 👉 TODO 4b: read_bytes += len(chunk)                  # count only what we read
        # 👉 TODO 4c: trace.append((pattern, f))                # record where it lives
        pass
    return trace, read_bytes

trace, read_bytes = incremental_trace()
for pattern, f in trace:
    print(f"  {pattern:24} -> {f}")
print("bytes read:", read_bytes)

**Self-check (offline).** All three functions traced, reading far less than the whole repo.

In [ ]:
# ===== self-check for TODO 4 =====
trace, read_bytes = incremental_trace()
assert len(trace) == 3, "should trace all three functions"
assert trace[2][1] == "talentdesk/candidates.py", "_lookup lives in candidates.py"
full_bytes = sum(len(p.read_text()) for p in Path(REPO).rglob("*") if p.is_file())
assert 0 < read_bytes < full_bytes, "incremental reads should be a fraction of the repo"
print(f"TODO 4 checks passed: read {read_bytes}/{full_bytes} bytes ({read_bytes/full_bytes:.0%})")

**This cell:** the **context-cost** comparison (provided): the incremental reads versus loading
the whole repo. Reading only the slices you need is what keeps the context window lean on a real
codebase.

In [ ]:
# ===== incremental reads vs loading everything (provided) =====
full_bytes = sum(len(p.read_text()) for p in Path(REPO).rglob("*") if p.is_file())
_, read_bytes = incremental_trace()
print("incremental bytes read:", read_bytes)
print("whole repo bytes:      ", full_bytes)
print(f"read {read_bytes / full_bytes:.0%} of the repo to trace the screening path")

**This cell:** the live run using the **real built-in tools** (provided). It points the tools at
the sample repo with `cwd=REPO` and allows Grep, Glob, Read, Edit, and Write. Ask it to change the bar
and it will Grep to find the line, Read the file, then Edit; ask it to change `return None` and it will
fall back to Write. Offline it prints the expected workflow.

In [ ]:
# ===== live: let Claude Code use the built-in tools =====
try:
    from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock, ToolUseBlock
    SDK_OK = True
    CODE_OPTS = ClaudeAgentOptions(model=MODEL, cwd=REPO,
                                   allowed_tools=["Grep", "Glob", "Read", "Edit", "Write"])
    async def ask(prompt):
        async for m in query(prompt=prompt, options=CODE_OPTS):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, ToolUseBlock): print("  ->", b.name, {k: b.input[k] for k in list(b.input)[:2]})
                    elif isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:140])
    print("built-in tools ready (cwd =", REPO, ")")
except Exception:
    SDK_OK = False
    print("Agent SDK not available offline; the tool analogues above were tested with Python.")

if RUN_LIVE and SDK_OK:
    print("--- unique-anchor edit ---")
    run_async(lambda: ask("In talentdesk/screening.py change MIN_YEARS to 4. Grep for it, Read the file, then Edit."))
    print("--- ambiguous anchor -> Read+Write fallback ---")
    run_async(lambda: ask("Make both 'return None' cases in screening.py return 'unknown candidate'."))
else:
    print("[offline] expected: Grep finds MIN_YEARS, Read loads screening.py, Edit sets it; then for")
    print("          the repeated 'return None', the agent uses Read then Write to change both.")

---

### Anti-patterns to avoid

| anti-pattern | what to do instead |
|---|---|
| Grep for a filename | use Glob for paths, Grep for contents |
| Edit with a short, common anchor | include enough surrounding context to make it unique |
| Edit without reading first | Read the file first (read-before-edit) |
| force a per-occurrence Edit on a repeated anchor | Read the file, transform in memory, Write it back |
| load the whole repo before starting | Grep to find, Read only the slice you need |
| use Write for a one-line change on a unique anchor | prefer Edit; reserve Write for the fallback |

**Lesson:** Claude Code's tools each do one job: **Grep** finds content, **Glob** finds files,
**Read** loads one file, **Edit** changes a unique anchor, **Write** replaces a whole file. Edit's
uniqueness rule is a feature: it stops the agent from changing the wrong line. When an anchor cannot be
made unique, fall back to **Read plus Write**; and explore **incrementally**, Grep to Read to trace, so
you pull in only the context the task needs.

---

## Recap - the built-in code tools

| Tool / habit | In this lab | Course topic |
|---|---|---|
| Grep | content by pattern | locating functions and usages (Lab 1) |
| Glob | file paths by pattern | discovering tests and config (Lab 1) |
| Edit | replace a unique anchor | targeted, surgical changes (Lab 1) |
| Read + Write fallback | change a repeated anchor safely | what Edit cannot do alone (Lab 2) |
| Incremental trace | Grep to Read to trace one slice at a time | keep the context window lean (Lab 2) |

**Try it next:** ask the agent to rename `meets_bar` and watch Edit need extra context to stay unique.
Then add a fourth module to the screening path and watch the incremental trace extend by exactly one
slice.